In [1]:
import pandas as pd
import numpy as np
import chainladder as cl

In [ ]:
raa_df = pd.read_csv(
    "https://raw.githubusercontent.com/casact/chainladder-python/master/chainladder/utils/data/raa.csv"
)
raa_df.head(20)

development    0
origin         0
values         0
dtype: int64

- Development: valuation date
- Origin: Accident year

In [4]:
raa_df['maturity'] = (raa_df['development'] - raa_df['origin'])*12 + 12

In [5]:
# We can observe the maturities like so
raa_df.head()

,development,origin,values,maturity
0,1981,1981,5012.0,12
1,1982,1982,106.0,12
2,1983,1983,3410.0,12
3,1984,1984,5655.0,12
4,1985,1985,1092.0,12


In [6]:
# NOTE Chain ladder arguments is not name appropriately
raa = cl.Triangle(
    raa_df,
    origin="origin",
    development="development",
    columns="values",
    cumulative= True
)
raa

,12,24,36,48,60,72,84,96,108,120
1981,"5,012","8,269","10,907","11,805","13,539","16,181","18,009","18,608","18,662","18,834"
1982,106,"4,285","5,396","10,666","13,782","15,599","15,496","16,169","16,704",
1983,"3,410","8,992","13,873","16,141","18,735","22,214","22,863","23,466",,
1984,"5,655","11,555","15,766","21,266","23,425","26,083","27,067",,,
1985,"1,092","9,565","15,836","22,169","25,955","26,180",,,,
1986,"1,513","6,445","11,702","12,935","15,852",,,,,
1987,557,"4,020","10,946","12,314",,,,,,
1988,"1,351","6,947","13,112",,,,,,,
1989,"3,133","5,395",,,,,,,,
1990,"2,063",,,,,,,,,


Not too well documented, but I see the point here, it just makes it convenient. A mapping that puts the correct values against the correct ones. I think the printing is using some sort of formatting. 

-  Figure out how the printing works here. Might be useful for showing outputs

In [7]:
raa = cl.load_sample('raa')
raa

,12,24,36,48,60,72,84,96,108,120
1981,"5,012","8,269","10,907","11,805","13,539","16,181","18,009","18,608","18,662","18,834"
1982,106,"4,285","5,396","10,666","13,782","15,599","15,496","16,169","16,704",
1983,"3,410","8,992","13,873","16,141","18,735","22,214","22,863","23,466",,
1984,"5,655","11,555","15,766","21,266","23,425","26,083","27,067",,,
1985,"1,092","9,565","15,836","22,169","25,955","26,180",,,,
1986,"1,513","6,445","11,702","12,935","15,852",,,,,
1987,557,"4,020","10,946","12,314",,,,,,
1988,"1,351","6,947","13,112",,,,,,,
1989,"3,133","5,395",,,,,,,,
1990,"2,063",,,,,,,,,


They provided a convenience function too. Might be useful to provide our users with example sets to perform these kinds of calculations.

In [8]:
raa.latest_diagonal

,1990
1981,"18,834"
1982,"16,704"
1983,"23,466"
1984,"27,067"
1985,"26,180"
1986,"15,852"
1987,"12,314"
1988,"13,112"
1989,"5,395"
1990,"2,063"


In [8]:
raa.valuation_date

Timestamp('1990-12-31 23:59:59.999999999')

In [9]:
print("Is triangle cumulative?", raa.is_cumulative)
print("Does triangle contain ultimate projections?", raa.is_ultimate)
print("Is this a valuation triangle?", raa.is_val_tri)
print('Has the triangle been "squared"?', raa.is_full)

Is triangle cumulative? True
Does triangle contain ultimate projections? False
Is this a valuation triangle? False
Has the triangle been "squared"? False


In [10]:
print("Origin grain:", raa.origin_grain)
print("Development grain:", raa.development_grain)

Origin grain: Y
Development grain: Y


The above refers to the data granualirity.

In [11]:
clrd_df = pd.read_csv(
    "https://raw.githubusercontent.com/casact/chainladder-python/master/chainladder/utils/data/clrd.csv"
)
clrd_df.head()

,GRCODE,GRNAME,AccidentYear,DevelopmentYear,DevelopmentLag,IncurLoss,CumPaidLoss,BulkLoss,EarnedPremDIR,EarnedPremCeded,EarnedPremNet,Single,PostedReserve97,LOB
0,86,Allstate Ins Co Grp,1988,1988,1,367404,70571,127737,400699,5957,394742,0,281872,wkcomp
1,86,Allstate Ins Co Grp,1988,1989,2,362988,155905,60173,400699,5957,394742,0,281872,wkcomp
2,86,Allstate Ins Co Grp,1988,1990,3,347288,220744,27763,400699,5957,394742,0,281872,wkcomp
3,86,Allstate Ins Co Grp,1988,1991,4,330648,251595,15280,400699,5957,394742,0,281872,wkcomp
4,86,Allstate Ins Co Grp,1988,1992,5,354690,274156,27689,400699,5957,394742,0,281872,wkcomp


In [12]:
clrd = cl.Triangle(
    clrd_df,
    origin="AccidentYear",
    development="DevelopmentYear",
    columns=[
        'IncurLoss',
        'CumPaidLoss',
        'BulkLoss',
        'EarnedPremDIR',
        'EarnedPremCeded',
        'EarnedPremNet',
    ],
    index=['GRNAME', 'LOB'],
    cumulative=True,
)

In [13]:
clrd

,Triangle Summary
Valuation:,1997-12
Grain:,OYDY
Shape:,"(775, 6, 10, 10)"
Index:,"[GRNAME, LOB]"
Columns:,"[IncurLoss, CumPaidLoss, BulkLoss, EarnedPremDIR, EarnedPremCeded, EarnedPremNet]"


- Valuation month is Dec 1997
- OYDY -> Origin (Accident) Year, Development Year
- Shape
    - 775: the unique combination of all possible indices
    - 6: Number of triangles (columns mentioned)
    - 10: Number of accident periods
    - 10: Number of valuation periods
- Index: Segmentation level

In [14]:
len((clrd_df['GRNAME'] +clrd_df['LOB']).unique())

775

In [15]:
clrd.index.head()

,GRNAME,LOB
0,Adriatic Ins Co,othliab
1,Adriatic Ins Co,ppauto
2,Aegis Grp,comauto
3,Aegis Grp,othliab
4,Aegis Grp,ppauto


In [16]:
np.sum(clrd.values)

np.float64(nan)

In [17]:
np.nansum(clrd.values)

np.float64(3661713596.0)

In [18]:
clrd[['CumPaidLoss', 'IncurLoss']]

,Triangle Summary
Valuation:,1997-12
Grain:,OYDY
Shape:,"(775, 2, 10, 10)"
Index:,"[GRNAME, LOB]"
Columns:,"[CumPaidLoss, IncurLoss]"


In [19]:
clrd[clrd["LOB"] == "wkcomp"]

,Triangle Summary
Valuation:,1997-12
Grain:,OYDY
Shape:,"(132, 6, 10, 10)"
Index:,"[GRNAME, LOB]"
Columns:,"[IncurLoss, CumPaidLoss, BulkLoss, EarnedPremDIR, EarnedPremCeded, EarnedPremNet]"


In [20]:
clrd.loc['Allstate Ins Co Grp']

,Triangle Summary
Valuation:,1997-12
Grain:,OYDY
Shape:,"(2, 6, 10, 10)"
Index:,[LOB]
Columns:,"[IncurLoss, CumPaidLoss, BulkLoss, EarnedPremDIR, EarnedPremCeded, EarnedPremNet]"


In [21]:
clrd.loc['Allstate Ins Co Grp'].index

,LOB
0,prodliab
1,wkcomp


In [22]:
clrd.loc['Allstate Ins Co Grp','wkcomp']

,Triangle Summary
Valuation:,1997-12
Grain:,OYDY
Shape:,"(1, 6, 10, 10)"
Index:,"[GRNAME, LOB]"
Columns:,"[IncurLoss, CumPaidLoss, BulkLoss, EarnedPremDIR, EarnedPremCeded, EarnedPremNet]"


In [23]:
clrd.loc['Allstate Ins Co Grp','wkcomp']['CumPaidLoss']

,12,24,36,48,60,72,84,96,108,120
1988,"70,571","155,905","220,744","251,595","274,156","287,676","298,499","304,873","321,808","325,322"
1989,"66,547","136,447","179,142","211,343","231,430","244,750","254,557","270,059","273,873",
1990,"52,233","133,370","178,444","204,442","222,193","232,940","253,337","256,788",,
1991,"59,315","128,051","169,793","196,685","213,165","234,676","239,195",,,
1992,"39,991","89,873","114,117","133,003","154,362","159,496",,,,
1993,"19,744","47,229","61,909","85,099","87,215",,,,,
1994,"20,379","46,773","88,636","91,077",,,,,,
1995,"18,756","84,712","87,311",,,,,,,
1996,"42,609","44,916",,,,,,,,
1997,691,,,,,,,,,


In [24]:
clrd['CaseIncurLoss'] = clrd['IncurLoss'] - clrd['BulkLoss']

In [25]:
clrd['PaidToIncurRatio'] = clrd['CumPaidLoss'] / clrd['IncurLoss']

Let's do some neat diagnostics

In [26]:
clrd.loc['Allstate Ins Co Grp','wkcomp']['PaidToIncurRatio']

,12,24,36,48,60,72,84,96,108,120
1988,0.1921,0.4295,0.6356,0.7609,0.7729,0.8217,0.8607,0.8733,0.9243,0.9355
1989,0.1975,0.4311,0.6432,0.6974,0.7727,0.8295,0.8506,0.9078,0.9110,
1990,0.1806,0.4283,0.6419,0.7361,0.8034,0.8377,0.9156,0.9135,,
1991,0.1996,0.4619,0.6295,0.7213,0.7857,0.8770,0.8872,,,
1992,0.2200,0.4382,0.5731,0.7081,0.8314,0.8624,,,,
1993,0.1720,0.4115,0.6102,0.8646,0.8998,,,,,
1994,0.1888,0.4348,0.9069,0.9469,,,,,,
1995,0.1863,0.8968,0.9458,,,,,,,
1996,0.7982,0.8772,,,,,,,,
1997,0.1028,,,,,,,,,


Loss Ratios here

In [27]:
clrd['CumPaidLoss'].sum() / clrd['EarnedPremNet'].sum()

,12,24,36,48,60,72,84,96,108,120
1988,0.2581,0.5093,0.6368,0.7115,0.7557,0.7802,0.7932,0.8002,0.8060,0.8083
1989,0.2686,0.5230,0.6526,0.7288,0.7727,0.7958,0.8085,0.8166,0.8204,
1990,0.2728,0.5249,0.6546,0.7287,0.7674,0.7888,0.8016,0.8080,,
1991,0.2540,0.4896,0.6093,0.6779,0.7152,0.7346,0.7453,,,
1992,0.2595,0.4927,0.6074,0.6711,0.7066,0.7245,,,,
1993,0.2645,0.4959,0.6061,0.6687,0.7021,,,,,
1994,0.2709,0.4942,0.6005,0.6614,,,,,,
1995,0.2651,0.4755,0.5737,,,,,,,
1996,0.2635,0.4586,,,,,,,,
1997,0.2552,,,,,,,,,


In [28]:
clrd.groupby("LOB").sum()

,Triangle Summary
Valuation:,1997-12
Grain:,OYDY
Shape:,"(6, 8, 10, 10)"
Index:,[LOB]
Columns:,"[IncurLoss, CumPaidLoss, BulkLoss, EarnedPremDIR, EarnedPremCeded, EarnedPremNet, CaseIncurLoss, PaidToIncurRatio]"


In [29]:
np.unique(clrd["LOB"])

array(['comauto', 'medmal', 'othliab', 'ppauto', 'prodliab', 'wkcomp'],
      dtype=object)

### Aggregation Methods

These don't explicitly need to apply to only the index.

In [30]:
clrd.sum(axis="index")

,Triangle Summary
Valuation:,1997-12
Grain:,OYDY
Shape:,"(1, 8, 10, 10)"
Index:,"[GRNAME, LOB]"
Columns:,"[IncurLoss, CumPaidLoss, BulkLoss, EarnedPremDIR, EarnedPremCeded, EarnedPremNet, CaseIncurLoss, PaidToIncurRatio]"


Summing with "index" as the axis means that we will get **eight** traingles with 10x10 AY and devlopments. But what do each of those triangles contain?

- "IncurLoss" will contain only summed up terms that come under the Incur loss category.
- The cell in "IncurLoss" that relates to 1990-36 corresponds to AY1990 and Dev of 36 months.
- This cell contains the sum of all values from each index (GRNAME and LOB combination) that corresponds to the Incurred Losses from AY1990 at 36 month maturity.

In [31]:
clrd.sum(axis="index")['IncurLoss']

,12,24,36,48,60,72,84,96,108,120
1988,"11,644,995","11,674,240","11,653,597","11,630,882","11,593,868","11,551,625","11,463,312","11,420,238","11,415,560","11,396,981"
1989,"13,123,290","13,118,789","13,113,024","13,050,144","12,959,037","12,866,709","12,787,372","12,757,420","12,743,440",
1990,"14,776,079","14,670,690","14,479,699","14,324,680","14,183,178","14,033,498","13,948,139","13,925,679",,
1991,"15,318,373","15,112,547","14,877,662","14,615,540","14,380,205","14,205,778","14,154,882",,,
1992,"16,828,857","16,457,307","15,999,385","15,538,214","15,249,286","15,161,066",,,,
1993,"18,169,370","17,590,902","17,080,187","16,485,467","16,281,774",,,,,
1994,"19,414,898","18,609,089","17,854,178","17,521,037",,,,,,
1995,"19,502,850","18,668,388","17,901,550",,,,,,,
1996,"19,142,090","17,910,743",,,,,,,,
1997,"18,113,581",,,,,,,,,


In [32]:
clrd.sum(axis="origin").sum(axis="index")["IncurLoss"]

,12,24,36,48,60,72,84,96,108,120
1988,"166,034,383","143,812,695","122,959,282","103,165,964","84,647,348","67,818,676","52,353,705","38,103,337","24,159,000","11,396,981"


Here we summed up all values from all accident years, though this is theoretically incorrect, as they should have been accounted for the trend differences.

## Accessor Methods

```python
# splits lastname from first name by a comma-delimiter
df['Last_First'].str.split(',')

# pulls the year out of each date in a dataframe column
df['Accident Date'].dt.year 
```

## Dates/Periods

In [33]:
clrd.origin

PeriodIndex(['1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995',
             '1996', '1997'],
            dtype='period[Y-DEC]', name='origin')

In [34]:
clrd.origin.min()

Period('1988', 'Y-DEC')

dtype=period(Year ending december)

In [35]:
clrd[clrd.origin == clrd.origin.min()].sum()['IncurLoss']
# summed over index

,12,24,36,48,60,72,84,96,108,120
1988,"11,644,995","11,674,240","11,653,597","11,630,882","11,593,868","11,551,625","11,463,312","11,420,238","11,415,560","11,396,981"


In [36]:
clrd[clrd.valuation > "1990"][clrd.origin <= "1995"].sum()['IncurLoss']

,12,24,36,48,60,72,84,96,108,120
1988,,,"11,653,597","11,630,882","11,593,868","11,551,625","11,463,312","11,420,238","11,415,560","11,396,981"
1989,,"13,118,789","13,113,024","13,050,144","12,959,037","12,866,709","12,787,372","12,757,420","12,743,440",
1990,"14,776,079","14,670,690","14,479,699","14,324,680","14,183,178","14,033,498","13,948,139","13,925,679",,
1991,"15,318,373","15,112,547","14,877,662","14,615,540","14,380,205","14,205,778","14,154,882",,,
1992,"16,828,857","16,457,307","15,999,385","15,538,214","15,249,286","15,161,066",,,,
1993,"18,169,370","17,590,902","17,080,187","16,485,467","16,281,774",,,,,
1994,"19,414,898","18,609,089","17,854,178","17,521,037",,,,,,
1995,"19,502,850","18,668,388","17,901,550",,,,,,,


## Link Ratios

In [37]:
clrd[(clrd.development > 12) & (clrd.development <= 36)].sum()['IncurLoss'].link_ratio

,24-36
1988,0.9982
1989,0.9996
1990,0.9870
1991,0.9845
1992,0.9722
1993,0.9710
1994,0.9594
1995,0.9589
1996,
1997,


In [38]:
clrd.sum()['IncurLoss'].link_ratio

,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
1988,1.0025,0.9982,0.9981,0.9968,0.9964,0.9924,0.9962,0.9996,0.9984
1989,0.9997,0.9996,0.9952,0.9930,0.9929,0.9938,0.9977,0.9989,
1990,0.9929,0.9870,0.9893,0.9901,0.9894,0.9939,0.9984,,
1991,0.9866,0.9845,0.9824,0.9839,0.9879,0.9964,,,
1992,0.9779,0.9722,0.9712,0.9814,0.9942,,,,
1993,0.9682,0.9710,0.9652,0.9876,,,,,
1994,0.9585,0.9594,0.9813,,,,,,
1995,0.9572,0.9589,,,,,,,
1996,0.9357,,,,,,,,


In [39]:
clrd[clrd['GRNAME'] == 'Alaska Nat Ins Co']['IncurLoss']

,Triangle Summary
Valuation:,1997-12
Grain:,OYDY
Shape:,"(4, 1, 10, 10)"
Index:,"[GRNAME, LOB]"
Columns:,[IncurLoss]


In [40]:
clrd.loc['Alaska Nat Ins Co','comauto']['EarnedPremNet']

,12,24,36,48,60,72,84,96,108,120
1988,"4,255","4,255","4,255","4,255","4,255","4,255","4,255","4,255","4,255","4,255"
1989,"4,178","4,178","4,178","4,178","4,178","4,178","4,178","4,178","4,178",
1990,"4,216","4,216","4,216","4,216","4,216","4,216","4,216","4,216",,
1991,"5,425","5,425","5,425","5,425","5,425","5,425","5,425",,,
1992,"5,835","5,835","5,835","5,835","5,835","5,835",,,,
1993,"6,170","6,170","6,170","6,170","6,170",,,,,
1994,"5,625","5,625","5,625","5,625",,,,,,
1995,"5,122","5,122","5,122",,,,,,,
1996,"4,973","4,973",,,,,,,,
1997,"4,308",,,,,,,,,


## Initializing $\triangle$ with own data

In [41]:
# ?cl.Triangle

In [ ]:
prism_df = pd.read_csv(
    "https://raw.githubusercontent.com/casact/chainladder-python/master/chainladder/utils/data/prism.csv"
)
prism_df.head()

,ClaimNo,AccidentDate,ReportDate,Line,Type,ClaimLiability,Limit,Deductible,TotalPayment,PaymentDate,CloseDate,Status,reportedCount,closedPaidCount,closedUnPaidCount,openCount,Paid,Outstanding,Incurred
0,1,2008-01-22,4/19/2010,Home,Dwelling,False,300000.0,20000,0.00000,2010-10-08,10/8/2010,CLOSED,1,0,1,0,0.00000,0,0.00000
1,2,2008-01-02,4/20/2010,Home,Dwelling,False,200000.0,20000,0.00000,2010-11-30,11/30/2010,CLOSED,1,0,1,0,0.00000,0,0.00000
2,3,2008-01-01,9/23/2009,Home,Dwelling,True,200000.0,20000,115744.77370,2010-02-17,2/17/2010,CLOSED,1,1,0,0,115744.77370,0,115744.77370
3,4,2008-01-02,7/25/2009,Home,Dwelling,True,200000.0,20000,63678.87713,2009-11-18,11/18/2009,CLOSED,1,1,0,0,63678.87713,0,63678.87713
4,5,2008-01-16,12/7/2009,Home,Dwelling,True,200000.0,20000,112175.55590,2010-04-30,4/30/2010,CLOSED,1,1,0,0,112175.55590,0,112175.55590


In [43]:
prism_df.dtypes

ClaimNo                int64
AccidentDate          object
ReportDate            object
Line                  object
Type                  object
ClaimLiability          bool
Limit                float64
Deductible             int64
TotalPayment         float64
PaymentDate           object
CloseDate             object
Status                object
reportedCount          int64
closedPaidCount        int64
closedUnPaidCount      int64
openCount              int64
Paid                 float64
Outstanding            int64
Incurred             float64
dtype: object

In [44]:
prism = cl.Triangle(
    data = prism_df,
    origin="AccidentDate",
    development="PaymentDate",
    columns="Paid",
    cumulative=False #(1)
)

1. Data is transactional

In [45]:
prism[(prism.origin <= '2008-05')][prism.development <= 4]

,1,2,3,4
2008-01,,,"46,915","19,899"
2008-02,,"28,749","22,109","79,033"
2008-03,,"48,806","27,949","90,413"
2008-04,,"30,758","17,763","70,872"
2008-05,,"38,672","86,974","20,483"


The given data actually is given in terms of accident years and payment period, and we can use python to view the data in terms of origin and development period. That's really neat as we just need to specify the correct columns. This is phenomenal. Meaning, we are allowed to load the **transactional** data in here directly and do manipulations to it as required. Finally, we can just put it in the `cl.Triangle` and viola! We have the completely ready-to-work triangle.

> NOTE: The minimum supported grain is monthly, but I think that's the point of using a triangle, to benefit from the averaging out higher and lower values within each period.

In [46]:
prism.origin_grain

'M'

In [47]:
prism.development_grain

'M'

In [48]:
prism[prism.origin >= '2017'].link_ratio

,1-2,2-3,3-4,4-5,5-6,6-7,7-8,8-9,9-10,10-11,11-12
2017-01,3.1183,4.6624,0.8591,1.0880,0.7598,1.6190,0.9113,0.5208,1.6887,0.7481,1.9370
2017-02,,2.0234,1.0180,0.9161,0.9946,1.6995,0.8849,0.9279,0.7163,0.6320,
2017-03,4.6412,3.5808,0.9712,1.9837,0.3929,3.7368,0.5563,0.8675,0.9166,,
2017-04,,4.1237,1.2211,1.0145,0.4590,1.4227,0.8915,1.5602,,,
2017-05,12.1797,1.4760,1.4804,1.0109,1.9780,1.1232,0.4896,,,,
2017-06,,1.1900,2.1426,1.4972,0.7004,1.2925,,,,,
2017-07,1.0749,4.0736,0.5769,1.0995,1.4910,,,,,,
2017-08,,7.5680,0.7738,2.1328,,,,,,,
2017-09,1.7923,2.5830,1.7679,,,,,,,,
2017-10,,2.9810,,,,,,,,,


In [49]:
prism = cl.Triangle(
    prism_df,
    origin="AccidentDate",
    development="PaymentDate",
    columns=["Paid", "Incurred"],
    cumulative=False,
)

In [50]:
prism

,Triangle Summary
Valuation:,2017-12
Grain:,OMDM
Shape:,"(1, 2, 120, 120)"
Index:,[Total]
Columns:,"[Paid, Incurred]"


In [51]:
prism_df.head()

,ClaimNo,AccidentDate,ReportDate,Line,Type,ClaimLiability,Limit,Deductible,TotalPayment,PaymentDate,CloseDate,Status,reportedCount,closedPaidCount,closedUnPaidCount,openCount,Paid,Outstanding,Incurred
0,1,2008-01-22,4/19/2010,Home,Dwelling,False,300000.0,20000,0.00000,2010-10-08,10/8/2010,CLOSED,1,0,1,0,0.00000,0,0.00000
1,2,2008-01-02,4/20/2010,Home,Dwelling,False,200000.0,20000,0.00000,2010-11-30,11/30/2010,CLOSED,1,0,1,0,0.00000,0,0.00000
2,3,2008-01-01,9/23/2009,Home,Dwelling,True,200000.0,20000,115744.77370,2010-02-17,2/17/2010,CLOSED,1,1,0,0,115744.77370,0,115744.77370
3,4,2008-01-02,7/25/2009,Home,Dwelling,True,200000.0,20000,63678.87713,2009-11-18,11/18/2009,CLOSED,1,1,0,0,63678.87713,0,63678.87713
4,5,2008-01-16,12/7/2009,Home,Dwelling,True,200000.0,20000,112175.55590,2010-04-30,4/30/2010,CLOSED,1,1,0,0,112175.55590,0,112175.55590


It is good practice to explicitly mention the date format.

In [52]:
prism = cl.Triangle(
    prism_df,
    origin="AccidentDate",
    origin_format="%Y-%m-%d",
    development="PaymentDate",
    development_format="%Y-%m-%d",
    columns=["Paid","Incurred"],
    cumulative=False
)

In [53]:
prism

,Triangle Summary
Valuation:,2017-12
Grain:,OMDM
Shape:,"(1, 2, 120, 120)"
Index:,[Total]
Columns:,"[Paid, Incurred]"


This works for symmetric triangles.

- Symmetric meaning having the same grain for both origin and development periods.

But what about asymmetric triangles?


In [54]:
prism_df['AccYr'] = prism_df['AccidentDate'].str[:4]

prism = cl.Triangle(
    prism_df,
    origin="AccYr",
    origin_format="%Y",
    development="PaymentDate",
    development_format="%Y-%m-%d",
    columns=["Paid","Incurred"],
    cumulative=False
)

In [55]:
prism

,Triangle Summary
Valuation:,2017-12
Grain:,OYDM
Shape:,"(1, 2, 10, 120)"
Index:,[Total]
Columns:,"[Paid, Incurred]"


In [56]:
prism['Paid']

,1,2,3,4,5,6,7,8,9,10,...,111,112,113,114,115,116,117,118,119,120
2008,,,"75,664","90,814","194,956","300,085","347,487","400,011","356,795","598,532",...,,,,"7,000",,,,,,
2009,,"24,327","80,376","136,857","175,737","297,690","358,624","325,462","529,832","540,217",...,,,,,,,,,,
2010,,"22,192","84,802","148,299","200,733","208,237","456,692","519,691","604,946","484,188",...,,,,,,,,,,
2011,,"24,934","60,404","107,198","233,346","268,904","401,595","530,490","501,836","513,133",...,,,,,,,,,,
2012,,"18,794","98,320","165,689","296,865","335,643","320,954","529,733","592,985","686,689",...,,,,,,,,,,
2013,,"70,959","99,628","238,991","180,138","302,213","477,224","585,963","648,734","637,203",...,,,,,,,,,,
2014,"18,194","32,772","124,801","276,336","344,635","425,875","449,242","541,203","587,789","819,083",...,,,,,,,,,,
2015,,"31,221","86,418","242,766","276,598","418,870","575,634","620,098","858,771","787,203",...,,,,,,,,,,
2016,"3,214","30,412","87,680","238,884","325,065","468,785","521,981","560,611","853,548","976,906",...,,,,,,,,,,
2017,"8,338","26,000","175,935","226,565","324,943","416,752","646,836","649,985","811,832","957,490",...,,,,,,,,,,


For a 1D triangle, simply omit the `development` argument

In [ ]:
prism_df["Premium"] = 3 * prism_df["Incurred"]

prism = cl.Triangle(
    prism_df,
    origin="AccidentDate",
    origin_format="%Y-%m-%d",
    columns="Premium",
    cumulative=False
)
prism.head()

,2017-12
2008-01,"41,872,294"
2008-02,"43,542,684"
2008-03,"35,920,802"
2008-04,"30,521,592"
2008-05,"46,807,611"
2008-06,"35,926,236"
2008-07,"33,305,612"
2008-08,"35,667,952"
2008-09,"36,773,174"
2008-10,"36,858,772"


Let's segment our data now

In [58]:
prism = cl.Triangle(
    prism_df,
    origin='AccidentDate',
    origin_format="%Y-%m-%d",
    development="PaymentDate",
    development_format="%Y-%m-%d",
    columns=["Paid","Incurred"],
    index=['Line','Type'],
    cumulative=False
)
prism

,Triangle Summary
Valuation:,2017-12
Grain:,OMDM
Shape:,"(2, 2, 120, 120)"
Index:,"[Line, Type]"
Columns:,"[Paid, Incurred]"


In [68]:
prism.index

,Line,Type
0,Auto,PD
1,Home,Dwelling


In [74]:
prism[prism['Line']=='Auto']['Paid']

,1,2,3,4,5,6,7,8,9,10,...,111,112,113,114,115,116,117,118,119,120
2008-01,,,"46,915","19,899","57,216","89,783","18,302","95,054","10,332","91,772",...,,,,,,,,,,
2008-02,,"28,749","22,109","79,033","63,455","59,993","64,683","68,502","33,695","71,670",...,,,,,,,,,,
2008-03,,"48,806","27,949","90,413","54,557","83,507","12,591","72,035","70,353","35,223",...,,,,,,,,,,
2008-04,,"30,758","17,763","70,872","30,233","39,494","66,729","122,100","20,998","38,218",...,,,,,,,,,,
2008-05,,"38,672","86,974","20,483","58,400","112,015","31,354","22,457","53,778","93,303",...,,,,,,,,,,
2008-06,,"56,789","73,351","97,840","64,816","61,083","41,685","87,172","61,418","63,097",...,,,,,,,,,,
2008-07,,"27,867","45,804","61,495","73,279","71,591","111,807","80,249","40,632","32,434",...,,,,,,,,,,
2008-08,"4,832","23,831","52,511","62,649","45,955","133,133","81,733","39,709","112,115","21,367",...,,,,,,,,,,
2008-09,,"43,464","86,157","52,664","89,928","88,738","55,617","26,599","69,522","46,788",...,,,,,,,,,,
2008-10,,"12,488","21,939","53,388","52,415","92,086","45,781","72,242","83,359","47,594",...,,,,,,,,,,


In [75]:
prism_cum = prism.incr_to_cum()
prism_cum[prism_cum['Line']=='Auto']['Paid']

,1,2,3,4,5,6,7,8,9,10,...,111,112,113,114,115,116,117,118,119,120
2008-01,,,"46,915","66,814","124,030","213,813","232,115","327,169","337,502","429,274",...,"1,178,976","1,178,976","1,178,976","1,178,976","1,178,976","1,178,976","1,178,976","1,178,976","1,178,976","1,178,976"
2008-02,,"28,749","50,859","129,891","193,346","253,339","318,022","386,523","420,218","491,888",...,"1,132,467","1,132,467","1,132,467","1,132,467","1,132,467","1,132,467","1,132,467","1,132,467","1,132,467",
2008-03,,"48,806","76,755","167,168","221,724","305,232","317,823","389,858","460,211","495,434",...,"1,416,496","1,416,496","1,416,496","1,416,496","1,416,496","1,416,496","1,416,496","1,416,496",,
2008-04,,"30,758","48,521","119,393","149,626","189,120","255,849","377,949","398,947","437,165",...,"1,014,483","1,014,483","1,014,483","1,014,483","1,014,483","1,014,483","1,014,483",,,
2008-05,,"38,672","125,646","146,129","204,529","316,543","347,897","370,354","424,131","517,435",...,"1,304,715","1,304,715","1,304,715","1,304,715","1,304,715","1,304,715",,,,
2008-06,,"56,789","130,140","227,980","292,796","353,879","395,565","482,736","544,154","607,251",...,"1,225,942","1,225,942","1,225,942","1,225,942","1,225,942",,,,,
2008-07,,"27,867","73,671","135,167","208,446","280,037","391,844","472,093","512,725","545,159",...,"1,202,380","1,202,380","1,202,380","1,202,380",,,,,,
2008-08,"4,832","28,663","81,174","143,824","189,779","322,912","404,645","444,354","556,469","577,837",...,"1,095,352","1,095,352","1,095,352",,,,,,,
2008-09,,"43,464","129,621","182,285","272,213","360,952","416,569","443,168","512,690","559,478",...,"1,280,106","1,280,106",,,,,,,,
2008-10,,"12,488","34,427","87,815","140,229","232,315","278,096","350,338","433,697","481,291",...,"1,249,236",,,,,,,,,


In [ ]:
# NOTE When we run the link_ratio() function, it just gives us the link ratio
# even if it is an incremental triangle (which is not what we usually expect).
# it is by design this way, so just keep in mind.

# prism.link_ratio['Paid'][prism['Line']=='Auto']
prism_cum.link_ratio['Paid'][prism['Line']=='Auto']

,1-2,2-3,3-4,4-5,5-6,6-7,7-8,8-9,9-10,10-11,...,110-111,111-112,112-113,113-114,114-115,115-116,116-117,117-118,118-119,119-120
2008-01,,,1.4241,1.8563,1.7239,1.0856,1.4095,1.0316,1.2719,1.1174,...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2008-02,,1.7690,2.5540,1.4885,1.3103,1.2553,1.2154,1.0872,1.1706,1.0220,...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,
2008-03,,1.5727,2.1779,1.3264,1.3766,1.0413,1.2267,1.1805,1.0765,1.1068,...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,,
2008-04,,1.5775,2.4606,1.2532,1.2640,1.3528,1.4772,1.0556,1.0958,1.0574,...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,,,
2008-05,,3.2490,1.1630,1.3996,1.5477,1.0991,1.0645,1.1452,1.2200,1.1298,...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,,,,
2008-06,,2.2916,1.7518,1.2843,1.2086,1.1178,1.2204,1.1272,1.1160,1.0329,...,1.0000,1.0000,1.0000,1.0000,1.0000,,,,,
2008-07,,2.6436,1.8347,1.5421,1.3435,1.3993,1.2048,1.0861,1.0633,1.0953,...,1.0000,1.0000,1.0000,1.0000,,,,,,
2008-08,5.9316,2.8320,1.7718,1.3195,1.7015,1.2531,1.0981,1.2523,1.0384,1.0336,...,1.0000,1.0000,1.0000,,,,,,,
2008-09,,2.9822,1.4063,1.4933,1.3260,1.1541,1.0639,1.1569,1.0913,1.2125,...,1.0000,1.0000,,,,,,,,
2008-10,,2.7569,2.5508,1.5969,1.6567,1.1971,1.2598,1.2379,1.1097,1.1015,...,1.0000,,,,,,,,,


In [82]:
prism_incr = prism_cum.cum_to_incr()

In [84]:
prism_incr.link_ratio['Paid'][prism['Line']=='Auto']

,1-2,2-3,3-4,4-5,5-6,6-7,7-8,8-9,9-10,10-11,...,110-111,111-112,112-113,113-114,114-115,115-116,116-117,117-118,118-119,119-120
2008-01,,,0.4241,2.8754,1.5692,0.2038,5.1937,0.1087,8.8820,0.5492,...,,,,,,,,,,
2008-02,,0.7690,3.5747,0.8029,0.9454,1.0782,1.0590,0.4919,2.1271,0.1513,...,,,,,,,,,,
2008-03,,0.5727,3.2349,0.6034,1.5307,0.1508,5.7209,0.9766,0.5007,1.5026,...,,,,,,,,,,
2008-04,,0.5775,3.9899,0.4266,1.3063,1.6896,1.8298,0.1720,1.8201,0.6567,...,,,,,,,,,,
2008-05,,2.2490,0.2355,2.8512,1.9181,0.2799,0.7162,2.3947,1.7350,0.7197,...,,,,,,,,,,
2008-06,,1.2916,1.3339,0.6625,0.9424,0.6824,2.0912,0.7046,1.0273,0.3171,...,,,,,,,,,,
2008-07,,1.6436,1.3426,1.1916,0.9770,1.5618,0.7177,0.5063,0.7982,1.6023,...,,,,,,,,,,
2008-08,4.9316,2.2035,1.1931,0.7335,2.8970,0.6139,0.4858,2.8234,0.1906,0.9073,...,,,,,,,,,,
2008-09,,1.9822,0.6113,1.7076,0.9868,0.6268,0.4783,2.6137,0.6730,2.5404,...,,,,,,,,,,
2008-10,,1.7569,2.4335,0.9818,1.7569,0.4972,1.5780,1.1539,0.5710,1.0263,...,,,,,,,,,,


We could have also done this

In [85]:
prism.incr_to_cum(inplace=True)

,Triangle Summary
Valuation:,2017-12
Grain:,OMDM
Shape:,"(2, 2, 120, 120)"
Index:,"[Line, Type]"
Columns:,"[Paid, Incurred]"


In [86]:
prism.is_cumulative

True

In [87]:
prism_OYDY = prism.grain('OYDY')

In [91]:
prism_OYDY[prism_OYDY['Line']=="Auto"]['Paid']

,12,24,36,48,60,72,84,96,108,120
2008,"3,404,254","10,061,780","12,954,138","13,822,198","14,182,320","14,352,172","14,428,318","14,464,252","14,476,664","14,483,664"
2009,"3,609,385","10,815,635","13,591,753","14,702,843","15,051,514","15,149,148","15,188,440","15,195,440","15,200,789",
2010,"4,067,321","11,776,175","15,127,586","16,292,386","16,884,254","17,029,935","17,102,366","17,112,956",,
2011,"4,125,232","12,679,848","16,191,719","17,441,295","17,936,011","18,165,842","18,239,063",,,
2012,"4,584,036","13,401,901","17,257,880","18,705,968","19,175,674","19,407,056",,,,
2013,"4,889,623","14,071,439","17,975,346","19,464,529","20,027,253",,,,,
2014,"5,546,158","15,442,153","19,862,529","21,427,083",,,,,,
2015,"5,909,029","17,056,596","21,867,472",,,,,,,
2016,"6,080,962","17,947,878",,,,,,,,
2017,"6,396,536",,,,,,,,,


Look at triangle with its `development` axis *expressed* as a valuation rather than an age.

In [94]:
prism_OYDY_val = prism_OYDY.dev_to_val()
prism_OYDY_val['Paid'].sum()

,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017
2008,"3,404,254","11,191,085","74,613,012","150,342,751","150,982,873","151,152,726","151,228,872","151,264,806","151,277,217","151,284,217"
2009,,"3,609,385","11,002,927","80,726,352","156,970,789","157,599,460","157,697,094","157,736,386","157,743,386","157,748,735"
2010,,,"4,067,321","12,396,777","74,210,043","161,049,586","161,641,453","161,787,135","161,859,565","161,870,156"
2011,,,,"4,125,232","13,183,144","81,239,771","161,412,913","162,187,629","162,417,460","162,490,681"
2012,,,,,"4,584,036","14,001,178","77,794,522","152,118,384","152,588,090","152,819,473"
2013,,,,,,"4,889,623","14,607,742","84,418,503","161,110,312","161,673,036"
2014,,,,,,,"5,546,158","16,408,126","77,256,792","154,969,931"
2015,,,,,,,,"5,909,029","17,427,611","80,914,580"
2016,,,,,,,,,"6,080,962","18,588,057"
2017,,,,,,,,,,"6,396,536"


In [97]:
prism_OYDY.loc['Home']['Paid']

,12,24,36,48,60,72,84,96,108,120
2008,,"1,129,305","61,658,874","136,520,554","136,800,554","136,800,554","136,800,554","136,800,554","136,800,554","136,800,554"
2009,,"187,292","67,134,599","142,267,946","142,547,946","142,547,946","142,547,946","142,547,946","142,547,946",
2010,,"620,603","59,082,456","144,757,200","144,757,200","144,757,200","144,757,200","144,757,200",,
2011,,"503,296","65,048,051","143,971,618","144,251,618","144,251,618","144,251,618",,,
2012,,"599,277","60,536,642","133,412,416","133,412,416","133,412,416",,,,
2013,,"536,303","66,443,157","141,645,782","141,645,782",,,,,
2014,,"965,973","57,394,263","133,542,848",,,,,,
2015,,"371,015","59,047,107",,,,,,,
2016,,"640,179",,,,,,,,
2017,,,,,,,,,,


In [98]:
prism_df_2011 = prism_df.copy()
prism_df_2011.loc[
    (prism_df_2011["AccidentDate"] >= "2011-01-01")
    & (prism_df_2011["AccidentDate"] < "2012-01-01"),
    "Paid",
] = None  # Let's assume we hav no payments for losses occurred in 2011
prism_2011 = cl.Triangle(
    data=prism_df_2011,
    origin="AccidentDate",
    origin_format="%Y-%m-%d",
    development="PaymentDate",
    development_format="%Y-%m-%d",
    columns=["Paid", "Incurred"],
    index=["Line", "Type"],
    cumulative=False,
)
prism_2011.incr_to_cum(inplace=True)
prism_2011_OYDY = prism_2011.grain("OYDY")
prism_2011_OYDY.loc["Home"]["Paid"]

,12,24,36,48,60,72,84,96,108,120
2008,,"1,129,305","61,658,874","136,520,554","136,800,554","136,800,554","136,800,554","136,800,554","136,800,554","136,800,554"
2009,,"187,292","67,134,599","142,267,946","142,547,946","142,547,946","142,547,946","142,547,946","142,547,946",
2010,,"620,603","59,082,456","144,757,200","144,757,200","144,757,200","144,757,200","144,757,200",,
2011,,,,,,,,,,
2012,,"599,277","60,536,642","133,412,416","133,412,416","133,412,416",,,,
2013,,"536,303","66,443,157","141,645,782","141,645,782",,,,,
2014,,"965,973","57,394,263","133,542,848",,,,,,
2015,,"371,015","59,047,107",,,,,,,
2016,,"640,179",,,,,,,,
2017,,,,,,,,,,


In [99]:
prism_2011_OYDY_droppedna = prism_2011_OYDY.loc["Home"].dropna()
prism_2011_OYDY_droppedna.loc["Home"]["Paid"]

,24,36,48,60,72,84,96,108,120
2008,"1,129,305","61,658,874","136,520,554","136,800,554","136,800,554","136,800,554","136,800,554","136,800,554","136,800,554"
2009,"187,292","67,134,599","142,267,946","142,547,946","142,547,946","142,547,946","142,547,946","142,547,946",
2010,"620,603","59,082,456","144,757,200","144,757,200","144,757,200","144,757,200","144,757,200",,
2011,,,,,,,,,
2012,"599,277","60,536,642","133,412,416","133,412,416","133,412,416",,,,
2013,"536,303","66,443,157","141,645,782","141,645,782",,,,,
2014,"965,973","57,394,263","133,542,848",,,,,,
2015,"371,015","59,047,107",,,,,,,
2016,"640,179",,,,,,,,


In [100]:
prism_OYDY.to_pickle('../data/prism_OYDY.pkl')

In [112]:
prism_OYDY_read = cl.read_pickle('../data/prism_OYDY.pkl')
prism_OYDY.sum(axis='index')['Paid']

,12,24,36,48,60,72,84,96,108,120
2008,"3,404,254","11,191,085","74,613,012","150,342,751","150,982,873","151,152,726","151,228,872","151,264,806","151,277,217","151,284,217"
2009,"3,609,385","11,002,927","80,726,352","156,970,789","157,599,460","157,697,094","157,736,386","157,743,386","157,748,735",
2010,"4,067,321","12,396,777","74,210,043","161,049,586","161,641,453","161,787,135","161,859,565","161,870,156",,
2011,"4,125,232","13,183,144","81,239,771","161,412,913","162,187,629","162,417,460","162,490,681",,,
2012,"4,584,036","14,001,178","77,794,522","152,118,384","152,588,090","152,819,473",,,,
2013,"4,889,623","14,607,742","84,418,503","161,110,312","161,673,036",,,,,
2014,"5,546,158","16,408,126","77,256,792","154,969,931",,,,,,
2015,"5,909,029","17,427,611","80,914,580",,,,,,,
2016,"6,080,962","18,588,057",,,,,,,,
2017,"6,396,536",,,,,,,,,


In [1]:
import chainladder as cl

In [2]:
benk = cl.Benktander()

In [5]:
raa = cl.load_sample('raa')
raa

,12,24,36,48,60,72,84,96,108,120
1981,"5,012","8,269","10,907","11,805","13,539","16,181","18,009","18,608","18,662","18,834"
1982,106,"4,285","5,396","10,666","13,782","15,599","15,496","16,169","16,704",
1983,"3,410","8,992","13,873","16,141","18,735","22,214","22,863","23,466",,
1984,"5,655","11,555","15,766","21,266","23,425","26,083","27,067",,,
1985,"1,092","9,565","15,836","22,169","25,955","26,180",,,,
1986,"1,513","6,445","11,702","12,935","15,852",,,,,
1987,557,"4,020","10,946","12,314",,,,,,
1988,"1,351","6,947","13,112",,,,,,,
1989,"3,133","5,395",,,,,,,,
1990,"2,063",,,,,,,,,


In [7]:
benk.fit(raa,sample_weight=raa)

,apriori,1.0
,n_iters,1
,apriori_sigma,0
,random_state,None
